##  Water Mask Extraction & Boundary Vectorization

Loads the cleaned SAR water mask from Notebook 4.

In [ ]:
import rasterio
import rasterio.features
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import shape

REGION_NAME = "naivasha"
IMAGE_ID = 20260228 
OUTPUT_DIR = f"./training/{REGION_NAME}"
MASK_PATH = f"{OUTPUT_DIR}/{REGION_NAME}_{IMAGE_ID}_water_mask.tif"

with rasterio.open(MASK_PATH) as src:
    water_mask = src.read(1)
    transform = src.transform
    crs = src.crs

assert not crs.is_geographic, "Mask must be in a projected CRS before any area/perimeter math"
print("CRS:", crs)
print("Water pixels:", water_mask.sum())

## 1. Raster → vector

`rasterio.features.shapes` walks the raster and yields `(geometry, value)` pairs for each contiguous region of equal value. We only want the regions where `value == 1` (water) — everything else (value 0, the background) gets discarded.

In [ ]:
shapes_gen = rasterio.features.shapes(water_mask.astype("uint8"), transform=transform)
geometries = [shape(geom) for geom, value in shapes_gen if value == 1]

print(f"Found {len(geometries)} raw water polygons")

gdf = gpd.GeoDataFrame({"geometry": geometries}, crs=crs)
gdf.plot(figsize=(6, 6), color="steelblue")
plt.title(f"Raw polygons ({len(gdf)})")
plt.axis("off")
plt.show()

Notice how many polygons there likely are compared to "one lake" — morphological cleanup in Notebook 4 removes a lot of noise, but small fragments still slip through. We filter those out next using area, not visual judgment, so the pipeline stays reproducible.

## 2. Area, perimeter, compactness

- **Area** and **perimeter** come directly from the geometry (in the projected CRS, so units are meters/m²) — this is *why* the CRS guard exists.
- **Compactness** = `4π·area / perimeter²`, ranges 0–1, where 1 is a perfect circle. A real lake boundary should be reasonably compact; a jagged sliver of misclassified pixels will have low compactness — useful for flagging junk polygons.

In [ ]:
gdf["area_m2"] = gdf.geometry.area
gdf["perimeter_m"] = gdf.geometry.length
gdf["compactness"] = (4 * np.pi * gdf["area_m2"]) / (gdf["perimeter_m"] ** 2)
gdf["compactness"] = gdf["compactness"].clip(0, 1)

gdf[["area_m2", "perimeter_m", "compactness"]].describe()

## 3. Filter small polygons, simplify

Drop polygons below a minimum area threshold (isolated noise, not real water), then simplify vertices slightly to reduce file size without meaningfully changing the shape.

In [ ]:
MIN_AREA_M2 = 20000  # ~2 hectares — tune per-region based on expected minimum feature size
SIMPLIFY_TOLERANCE = 10  # meters

original_count = len(gdf)
gdf = gdf[gdf["area_m2"] >= MIN_AREA_M2].reset_index(drop=True)
print(f"Kept {len(gdf)} / {original_count} polygons after area filter")

gdf["geometry"] = gdf.geometry.simplify(SIMPLIFY_TOLERANCE)

# Recompute area/perimeter/compactness — simplification changes them slightly
gdf["area_m2"] = gdf.geometry.area
gdf["perimeter_m"] = gdf.geometry.length
gdf["compactness"] = ((4 * np.pi * gdf["area_m2"]) / (gdf["perimeter_m"] ** 2)).clip(0, 1)

gdf.plot(figsize=(6, 6), color="steelblue", edgecolor="black")
plt.title(f"Filtered + simplified boundary ({len(gdf)} polygon(s))")
plt.axis("off")
plt.show()

## 4. Cross-check: SAR boundary vs optical-index boundary

SAR and optical indices are independent measurements of the same thing (open water). If they agree closely on the same date, that's good evidence the mask is trustworthy. If they diverge a lot, it's worth investigating — cloud contamination in the optical composite, or a SAR speckle/threshold issue.

We'll threshold the MNDWI raster from Notebook 3 the same way, using a simple fixed cutoff (MNDWI > 0 is a common convention), and vectorize it the same way as above.

In [ ]:
import rasterio as rio

# Recompute MNDWI locally from the S2 export (same approach as Notebook 3)
S2_PATH = f"{OUTPUT_DIR}/{REGION_NAME}_{IMAGE_ID}_s2.tif"

with rio.open(S2_PATH) as src:
    green = src.read(2).astype(float)
    swir1 = src.read(5).astype(float)
    s2_transform = src.transform
    s2_crs = src.crs

epsilon = 1e-10
mndwi = (green - swir1) / (green + swir1 + epsilon)
mndwi_water_mask = (mndwi > 0).astype("uint8")

mndwi_shapes = rasterio.features.shapes(mndwi_water_mask, transform=s2_transform)
mndwi_geoms = [shape(geom) for geom, value in mndwi_shapes if value == 1]
mndwi_gdf = gpd.GeoDataFrame({"geometry": mndwi_geoms}, crs=s2_crs)

if mndwi_gdf.crs.is_geographic:
    mndwi_gdf = mndwi_gdf.to_crs(mndwi_gdf.estimate_utm_crs())

mndwi_gdf["area_m2"] = mndwi_gdf.geometry.area
mndwi_gdf = mndwi_gdf[mndwi_gdf["area_m2"] >= MIN_AREA_M2].reset_index(drop=True)

sar_total_area = gdf["area_m2"].sum()
mndwi_total_area = mndwi_gdf["area_m2"].sum()
pct_diff = 100 * abs(sar_total_area - mndwi_total_area) / max(sar_total_area, mndwi_total_area)

print(f"SAR-derived water area:    {sar_total_area:,.0f} m²")
print(f"MNDWI-derived water area:  {mndwi_total_area:,.0f} m²")
print(f"Difference: {pct_diff:.1f}%")

In [ ]:
from matplotlib.lines import Line2D

fig, ax = plt.subplots(figsize=(7, 7))
gdf.to_crs(mndwi_gdf.crs).plot(ax=ax, color="none", edgecolor="blue", linewidth=2)
mndwi_gdf.plot(ax=ax, color="none", edgecolor="orange", linewidth=2, linestyle="--")

legend_handles = [
    Line2D([0], [0], color="blue", linewidth=2, label="SAR boundary"),
    Line2D([0], [0], color="orange", linewidth=2, linestyle="--", label="MNDWI boundary"),
]
ax.legend(handles=legend_handles)
ax.set_title("SAR vs MNDWI boundary — same date")
ax.axis("off")
plt.show()

A large disagreement here is usually diagnostic, not just noise: check cloud cover in the optical composite first (MNDWI is only as good as the cloud mask), then check the SAR threshold method before assuming one source is simply wrong.

## Exercise 5.1

Using your **Naivasha** SAR water mask from Notebook 4:

1. Vectorize it into a `GeoDataFrame`, filter by `MIN_AREA_M2`, and simplify.
2. Compute area, perimeter, compactness.
3. Recompute MNDWI locally from the Naivasha S2 export (Notebook 2/3) and vectorize it the same way.
4. Report the percent difference in total water area between the two sources, and give one hypothesis for the gap if it's larger than ~10%.

In [ ]:
# Your solution here
